In [ ]:
# P1-KN03: extract the selected OHR-Bench v2 PDFs from pdfs.zip -> output becomes dataset P1-KD03
import hashlib
import zipfile
from pathlib import Path

from huggingface_hub import hf_hub_download

SELECTED = [
    "academic/DUDE_1c6061aa2d4c3167592fff8e35e5100a", "academic/2310.11511v1", "academic/2405.14831v1",
    "administration/DUDE_0efb341dfb77bee6be16348a534eb9db", "administration/DUDE_eefcc78eed46846c09810b37f92e4cfb",
    "administration/DUDE_e79deb02a0c0e87511080836c5d4347b",
    "finance/AES_2022_10K", "finance/AMAZON_2017_10K", "finance/JPMORGAN_2023Q2_10Q",
    "law/ENERGYXXILTD_05_08_2015-EX-10.13-Transportation AGREEMENT",
    "law/ASIANDRAGONGROUPINC_08_11_2005-EX-10.5-Reseller Agreement",
    "law/MPLXLP_06_17_2015-EX-10.1-TRANSPORTATION SERVICES AGREEMENT",
    "manual/nova_y70", "manual/mi_phone", "manual/2021-Apple-Catalog",
]
OUTPUT = Path("/kaggle/working/ohr_selected")
OUTPUT.mkdir(parents=True, exist_ok=True)

archive = hf_hub_download("opendatalab/OHR-Bench", "pdfs.zip", repo_type="dataset", local_dir="/tmp/ohr")
with zipfile.ZipFile(archive) as bundle:
    pdf_names = [name for name in bundle.namelist()
                 if name.lower().endswith(".pdf") and not name.startswith("__MACOSX") and "/._" not in name]
    print("PDFs in archive:", len(pdf_names), "| example names:", pdf_names[:3])
    for doc_name in SELECTED:
        matches = [name for name in pdf_names if name == f"{doc_name}.pdf" or name.endswith(f"/{doc_name}.pdf")]
        if len(matches) != 1:
            print(f"  {doc_name}: {len(matches)} matches {matches} -> SKIPPED")
            continue
        content = bundle.read(matches[0])
        (OUTPUT / (doc_name.replace("/", "__") + ".pdf")).write_bytes(content)
        print(f"  {doc_name}: {len(content):,} bytes | sha256 {hashlib.sha256(content).hexdigest()[:16]}")